In [6]:
import os
import copy
import time
import random
import librosa
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit
)

from sklearn.preprocessing import (
    LabelEncoder
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    Dataset,
    DataLoader
)

#Next: Part 15: Save Outputs

# =====================================================
# SAVE PREDICTIONS
# =====================================================


import os
import copy
import time
import random
import librosa
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit
)

from sklearn.preprocessing import (
    LabelEncoder
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    Dataset,
    DataLoader
)

# =====================================================
# CONFIG
# =====================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using device:", DEVICE)

MANIFEST = "/home/feliciano/dataset_manifest.csv"

GROUP_SPLIT = "/home/feliciano/group_split.csv"

BACKGROUND_FOLDER = (
    "/media/feliciano/Aux/"
    "AI_AFS_DATASET/"
    "AFS_BEHAVIOUR_DATASET/"
    "background"
)

SAMPLE_RATE = 16000

N_MELS = 128

FMAX = 4000

BATCH_SIZE = 32

EPOCHS = 50

LEARNING_RATE = 1e-3

PATIENCE = 10


# =====================================================
# LOAD DATA
# =====================================================

fish_df = pd.read_csv(
    MANIFEST
)

split_df = pd.read_csv(
    GROUP_SPLIT
)

fish_df = fish_df.merge(
    split_df,
    on="parent_file_id",
    how="inner"
)

fish_df = fish_df[
    fish_df["label"]
    .str.lower()
    .isin(
        [
            "normal",
            "clustering",
            "agitation"
        ]
    )
].copy()

background_files = [

    os.path.join(
        BACKGROUND_FOLDER,
        f
    )

    for f in os.listdir(
        BACKGROUND_FOLDER
    )

    if f.lower().endswith(".wav")

]

background_df = pd.DataFrame({

    "clip_path":
        background_files,

    "label":
        "background"

})

bg_train, bg_test = train_test_split(

    background_df,

    test_size=0.20,

    random_state=SEED,

    shuffle=True

)

bg_train["split"] = "train"
bg_test["split"] = "test"

background_df = pd.concat(

    [
        bg_train,
        bg_test
    ],

    ignore_index=True

)

df = pd.concat(

    [
        fish_df,
        background_df
    ],

    ignore_index=True

)


# =====================================================
# LABELS
# =====================================================

encoder = LabelEncoder()

df["label_encoded"] = encoder.fit_transform(
    df["label"]
)

print("\nClasses:")

for i, cls in enumerate(
    encoder.classes_
):
    print(i, cls)


# =====================================================
# TRAIN TEST SPLIT
# =====================================================

train_df = df[
    df["split"] == "train"
].copy()

test_df = df[
    df["split"] == "test"
].copy()


# =====================================================
# VALIDATION SPLIT
# =====================================================

gss = GroupShuffleSplit(

    n_splits=1,

    test_size=0.10,

    random_state=SEED

)

train_idx, val_idx = next(

    gss.split(

        train_df,

        groups=np.arange(
            len(train_df)
        )

    )

)

train_final = train_df.iloc[
    train_idx
]

val_final = train_df.iloc[
    val_idx
]

print("Training:", len(train_final))
print("Validation:", len(val_final))
print("Testing:", len(test_df))

# =====================================================
# MEL EXTRACTION
# =====================================================

def extract_mel(path):

    signal, sr = librosa.load(

        path,

        sr=SAMPLE_RATE,

        mono=True

    )

    mel = librosa.feature.melspectrogram(

        y=signal,

        sr=sr,

        n_mels=N_MELS,

        fmax=FMAX

    )

    mel_db = librosa.power_to_db(

        mel,

        ref=np.max

    )

    return mel_db.astype(
        np.float32
    )




Using device: cuda

Classes:
0 agitation
1 background
2 clustering
3 normal
Training: 15479
Validation: 1720
Testing: 4500


In [8]:
# =====================================================
# DATASET
# =====================================================

class MelDataset(Dataset):

    def __init__(
        self,
        dataframe
    ):

        self.df = dataframe

    def __len__(self):

        return len(
            self.df
        )

    def __getitem__(
        self,
        idx
    ):

        row = self.df.iloc[idx]

        mel = extract_mel(
            row["clip_path"]
        )

        mel = torch.tensor(

            mel,

            dtype=torch.float32

        ).unsqueeze(0)

        label = torch.tensor(

            row["label_encoded"],

            dtype=torch.long

        )

        return (
            mel,
            label
        )


# =====================================================
# DATALOADERS
# =====================================================

train_ds = MelDataset(
    train_final
)

val_ds = MelDataset(
    val_final
)

test_ds = MelDataset(
    test_df
)

train_dl = DataLoader(

    train_ds,

    batch_size=BATCH_SIZE,

    shuffle=True

)

val_dl = DataLoader(

    val_ds,

    batch_size=BATCH_SIZE,

    shuffle=False

)

test_dl = DataLoader(

    test_ds,

    batch_size=BATCH_SIZE,

    shuffle=False

)

print(
    "\nTraining samples:",
    len(train_ds)
)

print(
    "Validation samples:",
    len(val_ds)
)

print(
    "Testing samples:",
    len(test_ds)
)


# =====================================================
# CLASS WEIGHTS
# =====================================================

class_counts = np.bincount(

    train_final[
        "label_encoded"
    ]

)

print(
    "\nClass counts:"
)

print(
    class_counts
)

weights = (

    len(train_final)

    /

    (

        len(class_counts)

        * class_counts

    )

)

weights = np.sqrt(
    weights
)

weights = torch.tensor(

    weights,

    dtype=torch.float32

).to(
    DEVICE
)

print(
    "\nClass weights:"
)

print(
    weights
)



# =====================================================
# FISHCNN OPTIMIZED
# =====================================================

class FishCNN_Optimized(
    nn.Module
):

    def __init__(
        self,
        num_classes=4
    ):

        super().__init__()

        # -----------------------------------------
        # BLOCK 1
        # -----------------------------------------

        self.block1 = nn.Sequential(

            nn.Conv2d(
                1,
                32,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(
                32
            ),

            nn.ReLU(),

            nn.Conv2d(
                32,
                32,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(
                32
            ),

            nn.ReLU(),

            nn.MaxPool2d(
                2
            )

        )

        # -----------------------------------------
        # BLOCK 2
        # -----------------------------------------

        self.block2 = nn.Sequential(

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(
                64
            ),

            nn.ReLU(),

            nn.Conv2d(
                64,
                64,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(
                64
            ),

            nn.ReLU(),

            nn.MaxPool2d(
                2
            )

        )

        # -----------------------------------------
        # BLOCK 3
        # -----------------------------------------

        self.block3 = nn.Sequential(

            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(
                128
            ),
original
            nn.ReLU(),

            nn.Conv2d(
                128,
                128,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(
                128
            ),

            nn.ReLU(),

            nn.MaxPool2d(
                2
            )

        )

        # -----------------------------------------
        # GLOBAL POOLING
        # -----------------------------------------

        self.gap = nn.AdaptiveAvgPool2d(
            (4, 4)
        )

        # -----------------------------------------
        # CLASSIFIER
        # -----------------------------------------

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                128 * 4 * 4,
                256
            ),

            nn.BatchNorm1d(
                256
            ),

            nn.ReLU(),

            nn.Dropout(
                0.40
            ),

            nn.Linear(
                256,
                128
            ),

            nn.BatchNorm1d(
                128
            ),

            nn.ReLU(),

            nn.Dropout(
                0.30
            ),

            nn.Linear(
                128,
                num_classes
            )

        )

    def forward(
        self,
        x
    ):

        x = self.block1(x)

        x = self.block2(x)

        x = self.block3(x)

        x = self.gap(x)

        x = self.classifier(x)

        return x


# =====================================================
# MODEL
# =====================================================

model = FishCNN_Optimized(

    num_classes=4

).to(
    DEVICE
)

criterion = nn.CrossEntropyLoss(

    weight=weights,

    label_smoothing=0.1

)

optimizer = optim.Adam(

    model.parameters(),

    lr=LEARNING_RATE

)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode="max",

    factor=0.5,

    patience=3

)

print("\nModel Initialised")

print(
    f"Classes: {len(encoder.classes_)}"
)

print(
    f"Device: {DEVICE}"
)



# =====================================================
# TRAINING
# =====================================================

best_f1 = 0.0

best_weights = None

patience_counter = 0

start_time = time.time()

print("\nTraining...\n")

for epoch in range(EPOCHS):

    # -----------------------------------------
    # TRAIN
    # -----------------------------------------

    model.train()

    running_loss = 0.0

    for mel, label in train_dl:

        mel = mel.to(
            DEVICE
        )

        label = label.to(
            DEVICE
        )

        optimizer.zero_grad()

        outputs = model(
            mel
        )

        loss = criterion(

            outputs,

            label

        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    train_loss = (
        running_loss
        /
        len(train_dl)
    )

    # -----------------------------------------
    # VALIDATION
    # -----------------------------------------

    model.eval()

    val_true = []

    val_pred = []

    with torch.no_grad():

        for mel, label in val_dl:

            mel = mel.to(
                DEVICE
            )

            outputs = model(
                mel
            )

            preds = torch.argmax(

                outputs,

                dim=1

            )

            val_true.extend(
                label.numpy()
            )

            val_pred.extend(
                preds.cpu().numpy()
            )

    val_f1 = f1_score(

        val_true,

        val_pred,

        average="macro"

    )

    scheduler.step(
        val_f1
    )

    current_lr = optimizer.param_groups[0]["lr"]

    print(

        f"Epoch {epoch+1:02d}/{EPOCHS} | "

        f"Loss={train_loss:.4f} | "

        f"Val_F1={val_f1:.4f} | "

        f"LR={current_lr:.6f}"

    )

    # -----------------------------------------
    # SAVE BEST MODEL
    # -----------------------------------------

    if val_f1 > best_f1:

        best_f1 = val_f1

        best_weights = copy.deepcopy(

            model.state_dict()

        )

        patience_counter = 0

    else:

        patience_counter += 1

    # -----------------------------------------
    # EARLY STOPPING
    # -----------------------------------------

    if patience_counter >= PATIENCE:

        print(
            "\nEarly stopping triggered."
        )

        break

training_time = (

    time.time()

    - start_time

)

print(
    f"\nTraining completed in {training_time:.2f} seconds"
)


# =====================================================
# LOAD BEST MODEL
# =====================================================

print(
    "\nLoading best model..."
)

model.load_state_dict(
    best_weights
)

# =====================================================
# TESTING
# =====================================================original

model.eval()

y_true = []
y_pred = []

start_inf = time.time()

with torch.no_grad():

    for mel, label in test_dl:

        mel = mel.to(
            DEVICE
        )

        outputs = model(
            mel
        )

        preds = torch.argmax(

            outputs,

            dim=1

        )

        y_true.extend(
            label.numpy()
        )

        y_pred.extend(
            preds.cpu().numpy()
        )

inference_time = (

    time.time()

    - start_inf

) / len(test_ds)

# =====================================================
# METRICS
# =====================================================

acc = accuracy_score(

    y_true,

    y_pred

)

prec = precision_score(

    y_true,

    y_pred,

    average="macro"

)

rec = recall_score(

    y_true,

    y_pred,

    average="macro"

)

f1 = f1_score(

    y_true,

    y_pred,

    average="macro"

)

cm = confusion_matrix(

    y_true,

    y_pred

)

print(
    "\n================================"
)

print(
    "OPTIMISED FISHCNN BACKGROUND"
)

print(
    "================================"
)

print(
    f"Accuracy : {acc:.4f}"
)

print(
    f"Precision: {prec:.4f}"
)

print(
    f"Recall   : {rec:.4f}"
)

print(
    f"F1       : {f1:.4f}"
)

print(
    f"Best Val : {best_f1:.4f}"
)

print(
    f"Training : {training_time:.2f}s"
)

print(
    f"Inference: {inference_time*1000:.4f} ms/sample"
)

print(
    "\nClassification Report\n"
)

print(

    classification_report(

        y_true,

        y_pred,

        target_names=
        encoder.classes_

    )

)

#Next: Part 15: Save Outputs

# =====================================================
# SAVE PREDICTIONS
# =====================================================


import os
import copy
import time
import random
import librosa
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit
)

from sklearn.preprocessing import (
    LabelEncoder
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,original
    confusion_matrix,
    classification_report
)

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    Dataset,
    DataLoader
)


pred_df = pd.DataFrame({

    "clip_path":import os
import copy
import time
import random
import librosa
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit
)

from sklearn.preprocessing import (
    LabelEncoder
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    Dataset,
    DataLoader
)
        test_df[
            "clip_path"
        ].values,

    "true_label":
        encoder.inverse_transform(
            np.array(y_true)
        ),

    "predicted_label":
        encoder.inverse_transform(
            np.array(y_pred)
        )

})

pred_df.to_csv(

    "fishcnn_optimised_background_predictions.csv",

    index=False

)

# =====================================================
# SAVE CONFUSION MATRIX
# =====================================================

pd.DataFrame(

    cm,

    index=encoder.classes_,

    columns=encoder.classes_

).to_csv(

    "fishcnn_optimised_background_cm.csv"

)

# =====================================================
# SAVE PER-CLASS F1
# =====================================================

per_class_f1 = f1_score(

    y_true,

    y_pred,

    average=None

)

pd.DataFrame({

    "class":
        encoder.classes_,

    "f1":
        per_class_f1

}).to_csv(

    "fishcnn_optimised_background_per_class_f1.csv",

    index=False

)

# =====================================================
# SAVE MODEL
# =====================================================

torch.save(

    model.state_dict(),

    "fishcnn_optimised_background.pth"

)

# =====================================================
# SAVE LABEL ENCODER
# =====================================================

joblib.dump(

    encoder,

    "fishcnn_optimised_background_encoder.pkl"

)

# =====================================================
# FINISHED
# =====================================================

print("\n================================")
print("FILES SAVED")
print("================================")

print(
    "fishcnn_optimised_background.pth"
)

print(
    "fishcnn_optimised_background_encoder.pkl"
)

print(
    "fishcnn_optimised_background_predictions.csv"
)

print(
    "fishcnn_optimised_background_cm.csv"
)

print(
    "fishcnn_optimised_background_per_class_f1.csv"
)

print("\nDone")


SyntaxError: invalid syntax. Perhaps you forgot a comma? (1177146224.py, line 282)